# Week 08 · Day 1 — RAG Basics: Chunking, Embeddings, Vector Databases, and Retrieval Evaluation


**A note on embeddings in this notebook:** a real setup would call a hosted embeddings endpoint (OpenAI, Cohere) or load a local neural model (`sentence-transformers`). This environment has no network access to either. So every "embedding" below is a **TF-IDF vector** — a classic, fully local, non-neural way to turn text into a vector. It gives you the same math as a real embedding (a fixed-length vector per piece of text, compared with cosine similarity, indexed in a real vector database) but it is fundamentally a **keyword-overlap** representation, not a true semantic one: two sentences only score high if they share literal words, not just related meaning. This matters and is called out explicitly at each step — including one place (Part C, step 5) where it changes the expected outcome of the kata in an instructive way. If you have access to a real embedding API or `sentence-transformers`, the only thing you'd swap is the `embed()` function below — everything downstream (Qdrant, hybrid search, precision/recall) is unchanged.


In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

def cosine_similarity(a, b):
    """Cosine similarity, computed by hand — the core formula from today's lesson."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


## Part A — Chunking and embeddings, by hand first

### A.1 — Chunk a real document two ways

Below is a short multi-paragraph "document" (a README-style explainer on Python virtual environments). We chunk it two ways:

- **Fixed-size, no overlap** — split every *N* words, no repetition between chunks.
- **Fixed-size, ~15% overlap** — each chunk repeats the tail of the previous one.

Then we find a real spot where the no-overlap version cuts an idea in half, and confirm the overlapping version keeps it intact.


In [2]:
doc = """Setting up a Python virtual environment is one of the first habits worth building as a developer. A virtual environment is an isolated Python installation that keeps the packages for one project separate from every other project on your machine. Without one, installing a new package for a side project can silently upgrade a library that a completely different project depends on, breaking it in ways that are hard to trace back to the real cause.

Creating one is simple: run python -m venv .venv in your project folder, then activate it with source .venv/bin/activate on macOS or Linux, or .venv\\Scripts\\activate on Windows. Once activated, any pip install command installs packages only inside that environment, not system-wide. You will know it worked because your shell prompt usually changes to show the environment's name in parentheses.

A common mistake beginners make is committing the .venv folder itself to version control. This bloats the repository with thousands of files that are specific to one machine's operating system and Python version, and will not work correctly if someone else clones the repo on a different platform. The fix is to add .venv to your .gitignore file and instead commit a requirements.txt or pyproject.toml that lists the packages needed, so anyone can recreate an equivalent environment with a single command.

Deactivating an environment is as simple as typing deactivate in the terminal. It's good practice to deactivate before switching to a different project, so you don't accidentally install a package meant for one project into the wrong environment. Over time, most developers end up with a small ritual: create a folder, create a venv, activate it, install dependencies, and only then start writing code."""

print(f"Document length: {len(doc.split())} words")


Document length: 280 words


In [3]:
# Manual Chunking
def chunk_fixed(text, chunk_size_words, overlap_ratio=0.0):
    """Split text into fixed-size word chunks, with an optional overlap ratio."""
    words = text.split()
    step = max(int(chunk_size_words * (1 - overlap_ratio)), 1)
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(" ".join(words[i:i + chunk_size_words]))
        if i + chunk_size_words >= len(words):
            break
        i += step
    return chunks

CHUNK_SIZE = 60          # ~ words per chunk (stand-in for tokens)
OVERLAP = 0.15           # 15%, matching the lesson's starting recommendation

no_overlap_chunks = chunk_fixed(doc, CHUNK_SIZE, overlap_ratio=0.0)
overlap_chunks = chunk_fixed(doc, CHUNK_SIZE, overlap_ratio=OVERLAP)

print(f"No-overlap chunks:  {len(no_overlap_chunks)}")
print(f"Overlap chunks:     {len(overlap_chunks)}")


No-overlap chunks:  5
Overlap chunks:     6


In [4]:
# Find the awkward cut: end of chunk 0 vs. start of chunk 1, no-overlap version
print("=== NO OVERLAP: the cut between chunk 0 and chunk 1 ===")
print("...chunk 0 ends with: ...", " ".join(no_overlap_chunks[0].split()[-12:]))
print("chunk 1 starts with:    ", " ".join(no_overlap_chunks[1].split()[:12]), "...")
print()
print("-> The idea 'a library that a completely different project depends on, breaking it...'")
print("   is severed right in the middle: 'project' ends chunk 0, 'depends on' starts chunk 1.")
print("   Read chunk 0 alone and the sentence is left hanging with no resolution.")


=== NO OVERLAP: the cut between chunk 0 and chunk 1 ===
...chunk 0 ends with: ... side project can silently upgrade a library that a completely different project
chunk 1 starts with:     depends on, breaking it in ways that are hard to trace back ...

-> The idea 'a library that a completely different project depends on, breaking it...'
   is severed right in the middle: 'project' ends chunk 0, 'depends on' starts chunk 1.
   Read chunk 0 alone and the sentence is left hanging with no resolution.


In [5]:
# Now the same boundary in the 15%-overlap version
print("=== WITH 15% OVERLAP: the same boundary ===")
print("...chunk 0 ends with: ...", " ".join(overlap_chunks[0].split()[-12:]))
print("chunk 1 starts with:    ", " ".join(overlap_chunks[1].split()[:15]), "...")
print()
print("-> chunk 1 now REPEATS the tail of chunk 0 ('...a completely different project')")
print("   before continuing into 'depends on, breaking it...' — so chunk 1 alone contains")
print("   the full idea intact. The overlap fixed exactly the cut we found above.")


=== WITH 15% OVERLAP: the same boundary ===
...chunk 0 ends with: ... side project can silently upgrade a library that a completely different project
chunk 1 starts with:     silently upgrade a library that a completely different project depends on, breaking it in ways ...

-> chunk 1 now REPEATS the tail of chunk 0 ('...a completely different project')
   before continuing into 'depends on, breaking it...' — so chunk 1 alone contains
   the full idea intact. The overlap fixed exactly the cut we found above.


**Takeaway (ties to lesson §3):** the no-overlap version proves the real cost of fixed-size chunking — an idea can be split exactly at an arbitrary word boundary. The 15% overlap doesn't prevent the *split* from happening, it just guarantees that at least one chunk contains the *whole* idea, so retrieval has a chance of surfacing it in one piece.

### A.2 — Compute cosine similarity by hand

Three sentences: two about the same topic (cooking), one unrelated (football). We embed them (TF-IDF, per the note above) and compute pairwise cosine similarity ourselves with `np.dot` / `np.linalg.norm` — no library cosine function.


In [6]:
s1 = "The chef grilled a salmon fillet and served it with lemon and herbs."
s2 = "Another chef grilled a salmon steak and finished it with fresh herbs."
s3 = "The quarterback threw a long touchdown pass in the fourth quarter."

vectorizer = TfidfVectorizer().fit([s1, s2, s3])
v1, v2, v3 = vectorizer.transform([s1, s2, s3]).toarray()

sim_related = cosine_similarity(v1, v2)      # both about cooking
sim_unrelated_1 = cosine_similarity(v1, v3)  # cooking vs. football
sim_unrelated_2 = cosine_similarity(v2, v3)  # cooking vs. football

print(f"sim(related pair, both cooking)      = {sim_related:.3f}")
print(f"sim(s1 cooking, s3 football)         = {sim_unrelated_1:.3f}")
print(f"sim(s2 cooking, s3 football)         = {sim_unrelated_2:.3f}")

assert sim_related > sim_unrelated_1 and sim_related > sim_unrelated_2
print("\nConfirmed: the related pair scores higher than either does against the unrelated sentence.")


sim(related pair, both cooking)      = 0.533
sim(s1 cooking, s3 football)         = 0.118
sim(s2 cooking, s3 football)         = 0.000

Confirmed: the related pair scores higher than either does against the unrelated sentence.


**Takeaway:** the related pair scores clearly higher (0.53) than either does against the football sentence (0.12 and 0.00). Cosine similarity is comparing *direction*, not sentence length — it doesn't care that `s1` and `s2` are slightly different lengths, only that they point the same way in vector space.

## Part B — A real vector database, end to end

### B.3 — Set up Qdrant locally (in-memory mode)

`qdrant-client` supports a fully in-process, in-memory mode with no server to run — perfect for today. We build a 10-sentence corpus across 5 topics (2 sentences each), embed them, and store each vector alongside its text as payload.


In [7]:
corpus = [
    "The chef grilled a salmon fillet and served it with lemon and herbs.",                         # food
    "Another chef grilled a salmon steak and finished it with fresh herbs.",                        # food
    "The quarterback threw a long touchdown pass in the fourth quarter.",                            # sports
    "In the fourth quarter, the team's quarterback threw another touchdown pass.",                  # sports
    "A Python virtual environment keeps a project's package dependencies isolated.",                 # tech
    "Isolating package dependencies in a Python virtual environment avoids version conflicts.",      # tech
    "Heavy rainfall this week raised river levels near the coastal town.",                           # weather
    "River levels near the coastal town kept rising after the heavy rainfall.",                      # weather
    "The museum's new exhibit features Renaissance paintings from Florence.",                        # art
    "A Renaissance painting exhibit from Florence opened at the museum.",                             # art
]
topics = ["food", "food", "sports", "sports", "tech", "tech", "weather", "weather", "art", "art"]

corpus_vectorizer = TfidfVectorizer().fit(corpus)
corpus_embeddings = corpus_vectorizer.transform(corpus).toarray().astype(np.float32)
embedding_dim = corpus_embeddings.shape[1]
print(f"Embedding dimension: {embedding_dim}")


Embedding dimension: 60


In [8]:
client = QdrantClient(":memory:")

client.create_collection(
    collection_name="sentences",
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
)

points = [
    PointStruct(id=i, vector=corpus_embeddings[i].tolist(), payload={"text": corpus[i], "topic": topics[i]})
    for i in range(len(corpus))
]
client.upsert(collection_name="sentences", points=points)
print(f"Stored {len(points)} points in the 'sentences' collection.")


Stored 10 points in the 'sentences' collection.


### B.4 — Query it and verify real retrieval

A query about a football play, phrased differently from either stored sports sentence, should retrieve the two sports sentences ahead of anything else — and a human reading the results should agree they're the right pick.


In [9]:
query = "A team's quarterback threw a touchdown pass late in the fourth quarter."
query_vector = corpus_vectorizer.transform([query]).toarray().astype(np.float32)[0]

hits = client.query_points(collection_name="sentences", query=query_vector.tolist(), limit=3).points

print(f"Query: {query}\n")
for h in hits:
    print(f"  score={h.score:.3f}  [{h.payload['topic']:8s}]  {h.payload['text']}")


Query: A team's quarterback threw a touchdown pass late in the fourth quarter.

  score=0.935  [sports  ]  In the fourth quarter, the team's quarterback threw another touchdown pass.
  score=0.829  [sports  ]  The quarterback threw a long touchdown pass in the fourth quarter.
  score=0.078  [tech    ]  Isolating package dependencies in a Python virtual environment avoids version conflicts.


**Result:** both sports sentences come back first (scores 0.94 and 0.83), well ahead of the nearest off-topic result (0.08). By inspection, that's exactly what a human would pick — the query is clearly about a football play, and that's what got retrieved.

## Part C — Hybrid search and evaluation, made concrete

### C.5 — Break semantic search on purpose

We add one sentence containing a fabricated ticket ID and query for the *exact ID string*, to see whether pure vector similarity search surfaces it reliably.


In [10]:
ticket_sentence = "Ticket REF-4471 was resolved by rotating the API key."
corpus_with_ticket = corpus + [ticket_sentence]

ticket_vectorizer = TfidfVectorizer().fit(corpus_with_ticket)
ticket_embeddings = ticket_vectorizer.transform(corpus_with_ticket).toarray().astype(np.float32)
ticket_dim = ticket_embeddings.shape[1]

ticket_client = QdrantClient(":memory:")
ticket_client.create_collection("sentences_with_ticket", vectors_config=VectorParams(size=ticket_dim, distance=Distance.COSINE))
ticket_points = [
    PointStruct(id=i, vector=ticket_embeddings[i].tolist(), payload={"text": corpus_with_ticket[i]})
    for i in range(len(corpus_with_ticket))
]
ticket_client.upsert(collection_name="sentences_with_ticket", points=ticket_points)

id_query = "REF-4471"
id_query_vector = ticket_vectorizer.transform([id_query]).toarray().astype(np.float32)[0]
hits = ticket_client.query_points(collection_name="sentences_with_ticket", query=id_query_vector.tolist(), limit=3).points

print(f"Exact-ID query: {id_query}\n")
for h in hits:
    print(f"  score={h.score:.3f}  {h.payload['text']}")


Exact-ID query: REF-4471

  score=0.466  Ticket REF-4471 was resolved by rotating the API key.
  score=0.000  A Renaissance painting exhibit from Florence opened at the museum.
  score=0.000  The museum's new exhibit features Renaissance paintings from Florence.


**Honest finding — and why it's the opposite of the usual case:** here, the vector search *does* find the ticket sentence reliably (top hit, score 0.47, with nothing else close). That's worth being honest about, but it's also worth explaining *why*, because it's a direct consequence of using TF-IDF instead of a real embedding model:

- TF-IDF is, structurally, a **keyword-overlap** method. `REF-4471` is a rare token that appears in exactly one sentence, so it gets a very high weight, and the query vector is almost entirely determined by that one dimension. Finding the one document containing it is trivial — this is essentially what keyword/BM25 search does well by design.
- A **real neural embedding model** behaves differently and typically *fails* this exact test: it has never seen `REF-4471` during training, has no learned representation for it, and produces a comparatively generic, low-signal vector for a bare code with no surrounding context — often failing to rank the ticket sentence above semantically-related-but-wrong results.

So the finding here (TF-IDF succeeds) doesn't contradict lesson §6 — it illustrates it from the other side. TF-IDF *is*, in effect, the keyword-search half of hybrid search, which is exactly why it doesn't fail on exact tokens the way a dense embedding does. If you swap in a real embedding API here, expect this exact same test to fail, which is precisely the case hybrid search (semantic + keyword, run in parallel) exists to fix — the keyword half would catch what the vector half misses.

### C.6 — Precision@3 and recall@3, by hand

Three queries against the Part B corpus, each with a known relevant set (the 2 sentences from the matching topic). For each, we retrieve the top 3 and compute precision@3 and recall@3 ourselves.


In [11]:
def precision_recall_at_k(hits, relevant_texts, k=3):
    top_k_texts = [h.payload["text"] for h in hits[:k]]
    n_relevant_retrieved = sum(1 for t in top_k_texts if t in relevant_texts)
    precision_at_k = n_relevant_retrieved / k
    recall_at_k = n_relevant_retrieved / len(relevant_texts)
    return precision_at_k, recall_at_k

eval_queries = [
    ("A chef grilled salmon and served it with lemon.", [corpus[0], corpus[1]]),
    ("Heavy rain caused river levels to rise near the coast.", [corpus[6], corpus[7]]),
    ("Keeping Python package dependencies isolated with a virtual environment.", [corpus[4], corpus[5]]),
]

for query_text, relevant in eval_queries:
    qv = corpus_vectorizer.transform([query_text]).toarray().astype(np.float32)[0]
    hits = client.query_points(collection_name="sentences", query=qv.tolist(), limit=3).points
    precision, recall = precision_recall_at_k(hits, relevant, k=3)

    print(f"Query: {query_text}")
    for h in hits:
        mark = "✓" if h.payload["text"] in relevant else " "
        print(f"  [{mark}] score={h.score:.3f}  {h.payload["text"]}")
    print(f"  precision@3 = {precision:.2f}   recall@3 = {recall:.2f}\n")


Query: A chef grilled salmon and served it with lemon.
  [✓] score=0.867  The chef grilled a salmon fillet and served it with lemon and herbs.
  [✓] score=0.581  Another chef grilled a salmon steak and finished it with fresh herbs.
  [ ] score=0.000  The museum's new exhibit features Renaissance paintings from Florence.
  precision@3 = 0.67   recall@3 = 1.00

Query: Heavy rain caused river levels to rise near the coast.
  [✓] score=0.634  River levels near the coastal town kept rising after the heavy rainfall.
  [✓] score=0.614  Heavy rainfall this week raised river levels near the coastal town.
  [ ] score=0.103  The quarterback threw a long touchdown pass in the fourth quarter.
  precision@3 = 0.67   recall@3 = 1.00

Query: Keeping Python package dependencies isolated with a virtual environment.
  [✓] score=0.777  A Python virtual environment keeps a project's package dependencies isolated.
  [✓] score=0.547  Isolating package dependencies in a Python virtual environment avoids versi

**Honest finding:** all three queries land at **precision@3 = 0.67** and **recall@3 = 1.00**. That's not a coincidence — it's a direct consequence of the setup: each topic has exactly 2 relevant sentences, both rank in the top 2 results every time (recall@3 = 2/2 = 1.0), and the corpus is small enough (10 sentences, 5 clean topics) that the 3rd retrieved slot is always something clearly off-topic, capping precision@3 at 2/3.

This is a useful, real number to sit with: **recall is perfect here only because the corpus is tiny and topic-separated.** At real scale — thousands of documents, topics that overlap, near-duplicate content — recall@k is exactly the metric that degrades first, and it's the one lesson §8 warns can't be fixed by better prompting: if the relevant chunk never makes it into the top-k retrieved set, the model never sees it, no matter how good it is at using what it *is* given.

## Summary — what today's kata actually demonstrated

| Step | Concept from the lesson | What we saw |
|---|---|---|
| A.1 | Chunking tradeoff | Fixed-size chunking cut a real sentence in half; 15% overlap kept it intact in one chunk |
| A.2 | Embeddings & cosine similarity | Same-topic sentences scored 0.53; unrelated scored 0.12 / 0.00 |
| B.3–B.4 | Vector databases | Qdrant in-memory index retrieved the right sentences for a paraphrased query |
| C.5 | Hybrid search  | Exact-ID query succeeded here *because* TF-IDF behaves like keyword search — the reverse case (a real neural embedding failing on the same query) is exactly why production systems run semantic + keyword search together |
| C.6 | Retrieval evaluation | precision@3 = 0.67, recall@3 = 1.00 on every query — and recall only looks this good because the corpus is small and cleanly separated by topic |
